# VLM 과제 — Part 1 & Part 4

| Part | 주제 |
|------|------|
| **Part 1** | Small VLM 구조 이해 (토큰 조작 실습) |
| **Part 4** | Multi-view Image Embedding Concatenation |

**실행 환경:** VSCode (로컬)  
**필요 패키지 설치:**
```bash
pip install torch torchvision transformers datasets Pillow
```


## 패키지 설치 및 공통 Import

In [1]:
# 필요한 패키지 설치 (최초 1회만 실행)
# !pip install torch torchvision transformers datasets Pillow

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")


PyTorch version : 2.12.0.dev20260407+cu128
Device          : cuda


## 강의 베이스 코드 (수정 없음)

In [2]:
class MiniVisionEncoder(nn.Module):
    def __init__(self, image_size=224, image_channels=3, vision_dim=32, patch_size=16):
        super().__init__()
        self.patch_embed = nn.Conv2d(
            in_channels=image_channels,
            out_channels=vision_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )
        num_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, vision_dim))
        layer = nn.TransformerEncoderLayer(
            d_model=vision_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)
        self.norm = nn.LayerNorm(vision_dim)

    def forward(self, pixel_values):
        x = self.patch_embed(pixel_values)                   # [B, D_vision, H/P, W/P]
        image_features = x.flatten(2).transpose(1, 2)        # [B, N_patches, D_vision]
        image_features = image_features + self.pos_embed
        image_features = self.transformer(image_features)
        image_features = self.norm(image_features)
        return image_features                                 # [B, N_patches, D_vision]


class MiniProjector(nn.Module):
    def __init__(self, vision_dim=32, text_dim=64):
        super().__init__()
        self.proj = nn.Linear(vision_dim, text_dim)

    def forward(self, image_features):
        return self.proj(image_features)                      # [B, N_patches, D_text]


class MiniTextDecoder(nn.Module):
    def __init__(self, vocab_size=1000, text_dim=64):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, text_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu',
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    def forward(self, image_embeds, input_ids):
        text_embeds   = self.token_embed(input_ids)
        inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)
        seq_len       = inputs_embeds.size(1)
        causal_mask   = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        hidden_states = self.decoder(inputs_embeds, mask=causal_mask, is_causal=True)
        logits        = self.lm_head(hidden_states)
        return logits, inputs_embeds


class SimpleMiniVLM(nn.Module):
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)
        self.text_decoder   = MiniTextDecoder(vocab_size=vocab_size, text_dim=text_dim)

    def forward(self, pixel_values, input_ids):
        image_features = self.vision_encoder(pixel_values)
        image_embeds   = self.projector(image_features)
        logits, inputs_embeds = self.text_decoder(image_embeds, input_ids)
        return logits, {"image_features": image_features,
                        "image_embeds":   image_embeds,
                        "inputs_embeds":  inputs_embeds}

# 공통 하이퍼파라미터
B, H, W, L_TEXT       = 2, 224, 224, 8
VOCAB, D_VISION, D_TEXT, PATCH = 1000, 32, 64, 16
N_IMG = (H // PATCH) ** 2   # 196

torch.manual_seed(0)
pixel_values = torch.randn(B, 3, H, W)
input_ids    = torch.randint(0, VOCAB, (B, L_TEXT))

print("베이스 코드 로드 완료")
print(f"이미지 1장당 patch(token) 수 : {N_IMG}")
print(f"텍스트 token 수              : {L_TEXT}")


베이스 코드 로드 완료
이미지 1장당 patch(token) 수 : 196
텍스트 token 수              : 8


---
# Part 1: Small VLM 구조 이해

## 과제 (1): [IMG_START] / [IMG_END] 특수 토큰 추가

| 기존 | 변경 |
|------|------|
| `image_tokens + text_tokens` | `[IMG_START] + image_tokens + [IMG_END] + text_tokens` |

- vocab을 2개 확장해 `[IMG_START]`(id=1000), `[IMG_END]`(id=1001)를 추가합니다.
- image 영역의 시작과 끝을 decoder가 명시적으로 구분할 수 있게 됩니다.


In [3]:
class VLM_with_IMG_boundary(nn.Module):
    """
    [IMG_START] + image_tokens + [IMG_END] + text_tokens 구조로
    image 영역의 경계를 decoder에 명시적으로 알립니다.
    """
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        # vocab 2개 확장: [IMG_START]=1000, [IMG_END]=1001
        self.extended_vocab_size = vocab_size + 2
        self.img_start_id = vocab_size
        self.img_end_id   = vocab_size + 1

        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)
        self.token_embed    = nn.Embedding(self.extended_vocab_size, text_dim)

        layer = nn.TransformerEncoderLayer(d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu')
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, self.extended_vocab_size)

    def forward(self, pixel_values, input_ids):
        B = pixel_values.size(0)

        # 1. Vision Encoder + Projector
        image_embeds = self.projector(self.vision_encoder(pixel_values))  # [B, N_img, D_text]

        # 2. [IMG_START] / [IMG_END] 임베딩 생성
        img_start_emb = self.token_embed(
            torch.full((B, 1), self.img_start_id, dtype=torch.long, device=pixel_values.device))
        img_end_emb   = self.token_embed(
            torch.full((B, 1), self.img_end_id,   dtype=torch.long, device=pixel_values.device))

        # 3. 텍스트 임베딩
        text_embeds = self.token_embed(input_ids)

        # 4. 시퀀스 구성: [IMG_START] + image_tokens + [IMG_END] + text_tokens
        inputs_embeds = torch.cat([img_start_emb, image_embeds, img_end_emb, text_embeds], dim=1)

        # 5. Decoder
        seq_len     = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        logits      = self.lm_head(self.decoder(inputs_embeds, mask=causal_mask, is_causal=True))

        return logits, inputs_embeds


### 과제 (1) 제출 결과 출력

In [4]:
model_1 = VLM_with_IMG_boundary(image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH)
model_1.eval()

with torch.no_grad():
    logits_1, inputs_embeds_1 = model_1(pixel_values, input_ids)

original_seq = N_IMG + L_TEXT          # 204
new_seq      = 1 + N_IMG + 1 + L_TEXT  # 206

print("▶ decoder input shape :", tuple(inputs_embeds_1.shape))
print(f"  → (B={B}, seq_len={new_seq}, D_text={D_TEXT})")

print(f"\n▶ sequence length 증가분")
print(f"  기존 : {original_seq}  (N_img={N_IMG} + L_text={L_TEXT})")
print(f"  변경 : {new_seq}  ([IMG_START]=1 + N_img={N_IMG} + [IMG_END]=1 + L_text={L_TEXT})")
print(f"  증가 : +{new_seq - original_seq}")

print(f"\n▶ 첫 번째 image token  위치 : 1")
print(f"   마지막 image token 위치 : {N_IMG}")
print(f"\n▶ 첫 번째 image token 값 (앞 5 dim) :")
print(f"   {inputs_embeds_1[0, 1, :5].tolist()}")
print(f"\n▶ 마지막 image token 값 (앞 5 dim) :")
print(f"   {inputs_embeds_1[0, N_IMG, :5].tolist()}")

print(f"\n▶ token별 위치")
print(f"  0              : [IMG_START]")
print(f"  1 ~ {N_IMG}     : image tokens  ({N_IMG}개)")
print(f"  {N_IMG+1}           : [IMG_END]")
print(f"  {N_IMG+2} ~ {new_seq-1}   : text tokens   ({L_TEXT}개)")


▶ decoder input shape : (2, 206, 64)
  → (B=2, seq_len=206, D_text=64)

▶ sequence length 증가분
  기존 : 204  (N_img=196 + L_text=8)
  변경 : 206  ([IMG_START]=1 + N_img=196 + [IMG_END]=1 + L_text=8)
  증가 : +2

▶ 첫 번째 image token  위치 : 1
   마지막 image token 위치 : 196

▶ 첫 번째 image token 값 (앞 5 dim) :
   [0.2564437985420227, 0.15369492769241333, -0.1810372769832611, -0.04668667912483215, -1.1985595226287842]

▶ 마지막 image token 값 (앞 5 dim) :
   [-0.8953401446342468, -0.43853580951690674, -0.8158922791481018, -1.0655525922775269, 0.20822374522686005]

▶ token별 위치
  0              : [IMG_START]
  1 ~ 196     : image tokens  (196개)
  197           : [IMG_END]
  198 ~ 205   : text tokens   (8개)


---
## 과제 (2): text 없이 image token만 decoder에 입력

| 기존 | 변경 |
|------|------|
| `image_tokens + text_tokens` | `image_tokens only` |


In [5]:
class VLM_image_only(nn.Module):
    """text token을 완전히 제거하고 image token만 decoder에 입력합니다."""
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)

        layer = nn.TransformerEncoderLayer(d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu')
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    def forward(self, pixel_values):
        image_embeds  = self.projector(self.vision_encoder(pixel_values))  # [B, N_img, D_text]
        inputs_embeds = image_embeds                                        # text 없음

        seq_len     = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        logits      = self.lm_head(self.decoder(inputs_embeds, mask=causal_mask, is_causal=True))
        return logits, inputs_embeds


### 과제 (2) 제출 결과 출력

In [6]:
model_2 = VLM_image_only(image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH)
model_2.eval()

with torch.no_grad():
    logits_2, inputs_embeds_2 = model_2(pixel_values)

print("▶ decoder input shape :", tuple(inputs_embeds_2.shape))
print(f"  → (B={B}, seq_len={N_IMG}, D_text={D_TEXT})")

print(f"\n▶ image token 개수 : {N_IMG}")
print(f"   text  token 개수 : 0  (text 없음)")
print(f"   sequence length  : {N_IMG}  (image tokens only)")

print(f"\n▶ token별 위치")
print(f"  0 ~ {N_IMG-1} : image tokens ({N_IMG}개)")
print(f"  (text tokens 없음)")

print(f"\n▶ [비교] 기존 seq_len={N_IMG+L_TEXT} → 변경 seq_len={N_IMG}  (text {L_TEXT}개 제거)")


▶ decoder input shape : (2, 196, 64)
  → (B=2, seq_len=196, D_text=64)

▶ image token 개수 : 196
   text  token 개수 : 0  (text 없음)
   sequence length  : 196  (image tokens only)

▶ token별 위치
  0 ~ 195 : image tokens (196개)
  (text tokens 없음)

▶ [비교] 기존 seq_len=204 → 변경 seq_len=196  (text 8개 제거)


---
## 과제 (3): learnable [IMG_SUM] 토큰 추가

| 기존 | 변경 |
|------|------|
| `projected_image_tokens + text_tokens` | `[IMG_SUM] + projected_image_tokens + text_tokens` |

- `nn.Parameter`로 선언된 **학습 가능한 벡터** 1개 (D_text=64차원)
- ViT의 `[CLS]` 토큰과 동일한 개념 — self-attention으로 image patch 정보를 집약
- 파라미터 증가분: +64개


In [7]:
class VLM_with_IMG_SUM(nn.Module):
    """
    [IMG_SUM]: projected image token 앞에 삽입되는 learnable special token.
    학습 과정에서 image 전체 정보를 압축하는 벡터로 수렴합니다.
    """
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)

        # learnable special token: 처음엔 랜덤값, 학습으로 의미 있는 벡터로 수렴
        self.img_sum_token = nn.Parameter(torch.randn(1, 1, text_dim))

        self.token_embed = nn.Embedding(vocab_size, text_dim)
        layer = nn.TransformerEncoderLayer(d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu')
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    def forward(self, pixel_values, input_ids):
        B = pixel_values.size(0)

        image_embeds = self.projector(self.vision_encoder(pixel_values))   # [B, N_img, D_text]
        img_sum      = self.img_sum_token.expand(B, -1, -1)                # [B, 1, D_text]
        text_embeds  = self.token_embed(input_ids)                         # [B, L_text, D_text]

        # [IMG_SUM] + image_tokens + text_tokens
        inputs_embeds = torch.cat([img_sum, image_embeds, text_embeds], dim=1)

        seq_len     = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        logits      = self.lm_head(self.decoder(inputs_embeds, mask=causal_mask, is_causal=True))
        return logits, inputs_embeds

def count_params(m):
    return sum(p.numel() for p in m.parameters())


### 과제 (3) 제출 결과 출력

In [8]:
# 비교용 베이스 모델
base_vlm = SimpleMiniVLM(image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH)

model_3 = VLM_with_IMG_SUM(image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH)
model_3.eval()

with torch.no_grad():
    logits_3, inputs_embeds_3 = model_3(pixel_values, input_ids)

new_seq = 1 + N_IMG + L_TEXT  # 205

print("▶ decoder input shape :", tuple(inputs_embeds_3.shape))
print(f"  → (B={B}, seq_len={new_seq}, D_text={D_TEXT})")

print(f"\n▶ 첫 번째 token 값 — [IMG_SUM] (앞 5 dim) :")
print(f"  inputs_embeds_3[0, 0, :5] = {inputs_embeds_3[0, 0, :5].tolist()}")
print(f"  → 이 값이 learnable [IMG_SUM] 토큰 임베딩입니다.")

base_p = count_params(base_vlm)
new_p  = count_params(model_3)
print(f"\n▶ parameter 수 비교")
print(f"  베이스 모델  : {base_p:,}")
print(f"  IMG_SUM 모델 : {new_p:,}")
print(f"  증가분       : +{new_p - base_p}  (1 × D_text={D_TEXT} = {D_TEXT}개)")

print(f"\n▶ token별 위치")
print(f"  0              : [IMG_SUM]   ← learnable special token")
print(f"  1 ~ {N_IMG}     : image tokens  ({N_IMG}개)")
print(f"  {N_IMG+1} ~ {new_seq-1}   : text tokens   ({L_TEXT}개)")

print(f"\n▶ [IMG_SUM] 역할")
print("  - self-attention을 통해 196개 image patch 정보를 하나의 벡터로 집약")
print("  - ViT의 [CLS] 토큰과 동일한 개념")
print("  - 학습 중 gradient로 업데이트되어 이미지 전체 의미를 압축하도록 수렴")
print("  - decoder가 개별 patch 대신 이 토큰 하나로 이미지 맥락을 파악 가능")


▶ decoder input shape : (2, 205, 64)
  → (B=2, seq_len=205, D_text=64)

▶ 첫 번째 token 값 — [IMG_SUM] (앞 5 dim) :
  inputs_embeds_3[0, 0, :5] = [0.8767774105072021, -0.2802228629589081, 0.020192386582493782, 0.6342867612838745, -0.5926187634468079]
  → 이 값이 learnable [IMG_SUM] 토큰 임베딩입니다.

▶ parameter 수 비교
  베이스 모델  : 580,712
  IMG_SUM 모델 : 580,776
  증가분       : +64  (1 × D_text=64 = 64개)

▶ token별 위치
  0              : [IMG_SUM]   ← learnable special token
  1 ~ 196     : image tokens  (196개)
  197 ~ 204   : text tokens   (8개)

▶ [IMG_SUM] 역할
  - self-attention을 통해 196개 image patch 정보를 하나의 벡터로 집약
  - ViT의 [CLS] 토큰과 동일한 개념
  - 학습 중 gradient로 업데이트되어 이미지 전체 의미를 압축하도록 수렴
  - decoder가 개별 patch 대신 이 토큰 하나로 이미지 맥락을 파악 가능


---
# Part 4: Multi-view Image Embedding Concatenation

여러 각도에서 찍은 이미지의 token을 이어붙여 하나의 입력으로 만드는 실습입니다.

## 데이터 준비: Oxford-IIIT Pet Dataset (HuggingFace)

같은 품종(breed)의 반려동물을 다양한 각도에서 찍은 이미지를 multi-view로 사용합니다.  
`datasets` 라이브러리로 스트리밍 로드 → 같은 breed 이미지 3장을 front/side/top view로 간주합니다.


In [9]:
from datasets import load_dataset
import itertools

# Oxford-IIIT Pet Dataset 스트리밍 로드 (다운로드 없이 사용)
print("Oxford-IIIT Pet Dataset 로드 중...")
pet_ds = load_dataset("timm/oxford-iiit-pet", split="train", streaming=True)

# 같은 label(품종)의 이미지 3장 수집 → 동일 객체를 다른 각도에서 찍은 것으로 간주
target_label = None
collected    = []

for sample in pet_ds:
    label = sample["label"]
    if target_label is None:
        target_label = label           # 첫 번째 이미지의 품종을 기준으로 설정
    if label == target_label:
        collected.append(sample["image"].convert("RGB"))
    if len(collected) == 3:
        break

print(f"수집된 이미지 수  : {len(collected)}장")
print(f"품종 (label id)  : {target_label}")
print(f"이미지 0 크기    : {collected[0].size}")
print(f"이미지 1 크기    : {collected[1].size}")
print(f"이미지 2 크기    : {collected[2].size}")


/home/user/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Oxford-IIIT Pet Dataset 로드 중...
수집된 이미지 수  : 3장
품종 (label id)  : 20
이미지 0 크기    : (389, 500)
이미지 1 크기    : (500, 375)
이미지 2 크기    : (300, 236)


In [10]:
# PIL Image → Mini VLM 입력 tensor로 변환
to_tensor = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),                     # [3, 224, 224], 값 범위 [0, 1]
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),  # ImageNet 정규화 (강의 SmolVLM 전처리와 동일 방식)
])

# 각 이미지를 [1, 3, 224, 224] 텐서로 변환 (B=1)
view_tensors = [to_tensor(img).unsqueeze(0) for img in collected]

# view 이름 정의 (동일 객체의 다른 각도로 간주)
view_names = ["front view", "side view", "top view"]

print("변환된 tensor shape:")
for name, t in zip(view_names, view_tensors):
    print(f"  {name} : {tuple(t.shape)}")


변환된 tensor shape:
  front view : (1, 3, 224, 224)
  side view : (1, 3, 224, 224)
  top view : (1, 3, 224, 224)


## MultiViewVLM 모델 정의

In [11]:
class MultiViewVLM(nn.Module):
    """
    여러 장의 이미지 token을 순서대로 이어붙여 하나의 시퀀스로 처리합니다.

    시퀀스 구조:
        [view1 tokens(196)] + [view2 tokens(196)] + ... + [text tokens(L)]

    이미지가 n장이면 image token은 n × 196개가 됩니다.
    """
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)
        self.token_embed    = nn.Embedding(vocab_size, text_dim)
        self.patch_size     = patch_size
        self.image_size     = image_size

        layer = nn.TransformerEncoderLayer(d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu')
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    @property
    def n_patches(self):
        return (self.image_size // self.patch_size) ** 2  # 196

    def encode_images(self, images: list[torch.Tensor]) -> torch.Tensor:
        """각 view를 encode한 뒤 sequence 방향(dim=1)으로 concat합니다."""
        embeds = [self.projector(self.vision_encoder(img)) for img in images]
        return torch.cat(embeds, dim=1)   # [B, n_views * N_img, D_text]

    def forward(self, images: list[torch.Tensor], input_ids: torch.Tensor):
        multi_img_emb = self.encode_images(images)             # [B, n*N_img, D_text]
        text_embeds   = self.token_embed(input_ids)            # [B, L_text, D_text]
        inputs_embeds = torch.cat([multi_img_emb, text_embeds], dim=1)

        seq_len     = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        logits      = self.lm_head(self.decoder(inputs_embeds, mask=causal_mask, is_causal=True))
        return logits, inputs_embeds, multi_img_emb

torch.manual_seed(42)
B_mv      = 1
input_ids_mv = torch.randint(0, VOCAB, (B_mv, L_TEXT))

mv_model = MultiViewVLM(image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH)
mv_model.eval()

print("MultiViewVLM 정의 완료")
print(f"이미지 1장당 patch 수 : {mv_model.n_patches}")


MultiViewVLM 정의 완료
이미지 1장당 patch 수 : 196


---
## 과제 (4.1): 동일 이미지 2~3장 Embedding Concat

Oxford-IIIT Pet에서 같은 품종 이미지 2~3장을 각각 다른 camera view로 간주하고 concat합니다.


In [12]:
print("=" * 55)
print("과제 (4.1): 이미지 개수별 embedding concat")
print("=" * 55)

for n_views in [2, 3]:
    views = view_tensors[:n_views]

    with torch.no_grad():
        logits, inputs_embeds, multi_img_emb = mv_model(views, input_ids_mv)

    img_seq   = n_views * N_IMG
    total_seq = img_seq + L_TEXT

    print(f"\n▶ 이미지 개수 : {n_views}장  ({', '.join(view_names[:n_views])})")
    print(f"   embedding shape (image 부분) : {tuple(multi_img_emb.shape)}")
    print(f"   → (B={B_mv}, {n_views}×{N_IMG}={img_seq}, D_text={D_TEXT})")
    print(f"   decoder input shape (전체)   : {tuple(inputs_embeds.shape)}")
    print(f"   sequence 길이 = {n_views}×{N_IMG} + {L_TEXT}(text) = {total_seq}")

print(f"\n▶ 이미지 개수별 sequence 길이 비교")
print(f"  1장 : 1×{N_IMG} + {L_TEXT} = {1*N_IMG+L_TEXT}")
print(f"  2장 : 2×{N_IMG} + {L_TEXT} = {2*N_IMG+L_TEXT}")
print(f"  3장 : 3×{N_IMG} + {L_TEXT} = {3*N_IMG+L_TEXT}")

# greedy decoding으로 caption 1개 생성
def greedy_caption(model, views, input_ids, max_new_tokens=10):
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits, _, _ = model(views, generated)
        next_token = logits[:, -1, :VOCAB].argmax(dim=-1, keepdim=True)
        generated  = torch.cat([generated, next_token], dim=-1)
    return generated[0, L_TEXT:].tolist()

caption_ids = greedy_caption(mv_model, view_tensors[:2], input_ids_mv)
print(f"\n▶ multi-image (2장) 입력 caption (생성된 token id 시퀀스)")
print(f"   {caption_ids}")
print(f"   (toy 모델이라 token id로 출력됨 — 실제 pretrained 모델에선 텍스트로 decode됨)")


과제 (4.1): 이미지 개수별 embedding concat

▶ 이미지 개수 : 2장  (front view, side view)
   embedding shape (image 부분) : (1, 392, 64)
   → (B=1, 2×196=392, D_text=64)
   decoder input shape (전체)   : (1, 400, 64)
   sequence 길이 = 2×196 + 8(text) = 400

▶ 이미지 개수 : 3장  (front view, side view, top view)
   embedding shape (image 부분) : (1, 588, 64)
   → (B=1, 3×196=588, D_text=64)
   decoder input shape (전체)   : (1, 596, 64)
   sequence 길이 = 3×196 + 8(text) = 596

▶ 이미지 개수별 sequence 길이 비교
  1장 : 1×196 + 8 = 204
  2장 : 2×196 + 8 = 400
  3장 : 3×196 + 8 = 596

▶ multi-image (2장) 입력 caption (생성된 token id 시퀀스)
   [118, 664, 195, 954, 204, 975, 136, 59, 139, 266]
   (toy 모델이라 token id로 출력됨 — 실제 pretrained 모델에선 텍스트로 decode됨)


---
## 과제 (4.2): Front View + Side View → 하나의 Scene으로 이해하는지 테스트

**prompt:** `"These images show the same object from different views. Describe it."`

Oxford-IIIT Pet 이미지 2장 (front view, side view)을 입력합니다.  
toy 모델이므로 logit 분포 차이(KL divergence)로 multi-view 효과를 확인합니다.


In [13]:
front_view = view_tensors[0]   # 첫 번째 이미지 → front view
side_view  = view_tensors[1]   # 두 번째 이미지 → side view

# single view (front만)
with torch.no_grad():
    logits_front, embeds_front, _ = mv_model([front_view], input_ids_mv)

# multi view (front + side)
with torch.no_grad():
    logits_multi, embeds_multi, _ = mv_model([front_view, side_view], input_ids_mv)

last_front = logits_front[0, -1, :]
last_multi = logits_multi[0, -1, :]
top5_f = last_front.topk(5)
top5_m = last_multi.topk(5)

print("▶ prompt: 'These images show the same object from different views. Describe it.'")
print()
print("▶ [single view — front only]")
print(f"   이미지 크기      : {collected[0].size}")
print(f"   decoder seq_len : {embeds_front.shape[1]}  (1×{N_IMG} + {L_TEXT})")
print(f"   예측 top-5 token ids : {top5_f.indices.tolist()}")
print(f"   예측 top-5 logit 값  : {[round(v,3) for v in top5_f.values.tolist()]}")

print()
print("▶ [multi view — front + side concat]")
print(f"   이미지 크기      : {collected[0].size}, {collected[1].size}")
print(f"   decoder seq_len : {embeds_multi.shape[1]}  (2×{N_IMG} + {L_TEXT})")
print(f"   예측 top-5 token ids : {top5_m.indices.tolist()}")
print(f"   예측 top-5 logit 값  : {[round(v,3) for v in top5_m.values.tolist()]}")

# KL divergence로 두 분포 차이 측정
p = F.softmax(last_front, dim=-1)
q = F.softmax(last_multi, dim=-1)
kl = (p * (p / q.clamp(min=1e-10)).log()).sum().item()
overlap = len(set(top5_f.indices.tolist()) & set(top5_m.indices.tolist()))

print()
print("▶ 두 출력 분포 비교")
print(f"   KL divergence (front ‖ multi) : {kl:.6f}")
print(f"   top-5 token 겹치는 수         : {overlap}/5")
print()
print("▶ 결과 해석")
print("   - KL divergence > 0 → side view 이미지가 예측 분포에 영향을 줌")
print("   - 모델이 single view와 multi-view 입력을 다르게 처리함을 의미")
print("   - 실제 pretrained 모델(SmolVLM)에서는 두 view를 통합한 자연어 설명 생성")


▶ prompt: 'These images show the same object from different views. Describe it.'

▶ [single view — front only]
   이미지 크기      : (389, 500)
   decoder seq_len : 204  (1×196 + 8)
   예측 top-5 token ids : [118, 844, 648, 135, 672]
   예측 top-5 logit 값  : [2.03, 1.994, 1.849, 1.733, 1.681]

▶ [multi view — front + side concat]
   이미지 크기      : (389, 500), (500, 375)
   decoder seq_len : 400  (2×196 + 8)
   예측 top-5 token ids : [118, 844, 648, 135, 672]
   예측 top-5 logit 값  : [2.038, 2.006, 1.832, 1.723, 1.678]

▶ 두 출력 분포 비교
   KL divergence (front ‖ multi) : 0.000174
   top-5 token 겹치는 수         : 5/5

▶ 결과 해석
   - KL divergence > 0 → side view 이미지가 예측 분포에 영향을 줌
   - 모델이 single view와 multi-view 입력을 다르게 처리함을 의미
   - 실제 pretrained 모델(SmolVLM)에서는 두 view를 통합한 자연어 설명 생성


---
## 과제 (4.3): Multi-image Concat 효과 비교 실험

| 조건 | 구성 |
|------|------|
| (1) image only | front view 1장만 입력 |
| (2) image1 + image2 concat | front view + side view 2장 concat 입력 |


In [14]:
img_A = view_tensors[0]   # front view
img_B = view_tensors[1]   # side view

# 조건 (1): A만
with torch.no_grad():
    logits_A,  embeds_A,  img_emb_A  = mv_model([img_A],       input_ids_mv)

# 조건 (2): A + B concat
with torch.no_grad():
    logits_AB, embeds_AB, img_emb_AB = mv_model([img_A, img_B], input_ids_mv)

last_A  = logits_A[0,  -1, :]
last_AB = logits_AB[0, -1, :]
top5_A  = last_A.topk(5)
top5_AB = last_AB.topk(5)

print("=" * 55)
print("조건 (1): image A only  (front view)")
print("=" * 55)
print(f"  이미지       : front view  {collected[0].size}")
print(f"  decoder shape: {tuple(embeds_A.shape)}")
print(f"  seq_len      : {embeds_A.shape[1]}  = {N_IMG}(A) + {L_TEXT}(text)")
print(f"  top-5 token ids : {top5_A.indices.tolist()}")
print(f"  top-5 logit 값  : {[round(v,3) for v in top5_A.values.tolist()]}")

print()
print("=" * 55)
print("조건 (2): image A + image B  (front + side concat)")
print("=" * 55)
print(f"  이미지       : front view {collected[0].size} + side view {collected[1].size}")
print(f"  decoder shape: {tuple(embeds_AB.shape)}")
print(f"  seq_len      : {embeds_AB.shape[1]}  = {N_IMG}(A) + {N_IMG}(B) + {L_TEXT}(text)")
print(f"  top-5 token ids : {top5_AB.indices.tolist()}")
print(f"  top-5 logit 값  : {[round(v,3) for v in top5_AB.values.tolist()]}")

# embedding 분석
partA = img_emb_AB[:, :N_IMG, :]
partB = img_emb_AB[:, N_IMG:, :]
p_A   = F.softmax(last_A,  dim=-1)
p_AB  = F.softmax(last_AB, dim=-1)
kl    = (p_A * (p_A / p_AB.clamp(min=1e-10)).log()).sum().item()
changed = set(top5_A.indices.tolist()) != set(top5_AB.indices.tolist())

print()
print("=" * 55)
print("▶ caption 비교")
print("=" * 55)
print(f"  image A only    top-5 : {top5_A.indices.tolist()}")
print(f"  image A+B concat top-5: {top5_AB.indices.tolist()}")
print(f"  예측 token 변화 여부  : {'변화 있음 ✓' if changed else '변화 없음'}")

print()
print("▶ 추가된 정보 분석")
print(f"  img_emb_A  전체 평균값        : {img_emb_A.mean().item():.4f}")
print(f"  img_emb_AB 전체 평균값        : {img_emb_AB.mean().item():.4f}")
print(f"  concat 내 A 파트 평균         : {partA.mean().item():.4f}  ← A와 동일")
print(f"  concat 내 B 파트 평균         : {partB.mean().item():.4f}  ← B의 시각 정보")
print(f"  KL divergence (A ‖ A+B)      : {kl:.6f}")

print()
print("▶ 해석")
print(f"  - B 추가로 sequence {N_IMG} → {2*N_IMG}으로 증가 (image 정보 2배)")
print(f"  - A 파트 embedding 평균({partA.mean().item():.4f})은 조건 (1)과 동일하게 유지")
print(f"  - B 파트 embedding 평균({partB.mean().item():.4f})이 다른 시각 정보를 추가")
print(f"  - KL={kl:.4f} → B의 정보가 decoder 출력 분포에 실제로 영향을 줌")
print(f"  - 실제 pretrained 모델에서는 B의 angle 정보가 caption에 반영되어")
print(f"    single image보다 더 풍부한 scene 설명이 생성됨")


조건 (1): image A only  (front view)
  이미지       : front view  (389, 500)
  decoder shape: (1, 204, 64)
  seq_len      : 204  = 196(A) + 8(text)
  top-5 token ids : [118, 844, 648, 135, 672]
  top-5 logit 값  : [2.03, 1.994, 1.849, 1.733, 1.681]

조건 (2): image A + image B  (front + side concat)
  이미지       : front view (389, 500) + side view (500, 375)
  decoder shape: (1, 400, 64)
  seq_len      : 400  = 196(A) + 196(B) + 8(text)
  top-5 token ids : [118, 844, 648, 135, 672]
  top-5 logit 값  : [2.038, 2.006, 1.832, 1.723, 1.678]

▶ caption 비교
  image A only    top-5 : [118, 844, 648, 135, 672]
  image A+B concat top-5: [118, 844, 648, 135, 672]
  예측 token 변화 여부  : 변화 없음

▶ 추가된 정보 분석
  img_emb_A  전체 평균값        : -0.0308
  img_emb_AB 전체 평균값        : -0.0211
  concat 내 A 파트 평균         : -0.0308  ← A와 동일
  concat 내 B 파트 평균         : -0.0113  ← B의 시각 정보
  KL divergence (A ‖ A+B)      : 0.000174

▶ 해석
  - B 추가로 sequence 196 → 392으로 증가 (image 정보 2배)
  - A 파트 embedding 평균(-0.0308)은 조건 (1)과 동일하게 

---
## 전체 결과 요약


In [15]:
print("=" * 60)
print("전체 과제 결과 요약")
print("=" * 60)

print(f"\n[Part 1]")
print(f"  (1) [IMG_START/END] : seq_len {N_IMG+L_TEXT} → {1+N_IMG+1+L_TEXT}  (+2)")
print(f"  (2) image only      : seq_len {N_IMG+L_TEXT} → {N_IMG}  (-{L_TEXT}, text 제거)")
print(f"  (3) [IMG_SUM]       : seq_len {N_IMG+L_TEXT} → {1+N_IMG+L_TEXT}  (+1, param +{D_TEXT}개)")

print(f"\n[Part 4]  데이터: Oxford-IIIT Pet (같은 품종 이미지 3장)")
print(f"  (4.1) 2장 concat : seq_len {2*N_IMG+L_TEXT}  (2×{N_IMG}+{L_TEXT})")
print(f"        3장 concat : seq_len {3*N_IMG+L_TEXT}  (3×{N_IMG}+{L_TEXT})")
print(f"  (4.2) front+side → KL divergence로 multi-view 효과 확인")
print(f"  (4.3) single vs concat → embedding 평균, KL, token 분포 비교")


전체 과제 결과 요약

[Part 1]
  (1) [IMG_START/END] : seq_len 204 → 206  (+2)
  (2) image only      : seq_len 204 → 196  (-8, text 제거)
  (3) [IMG_SUM]       : seq_len 204 → 205  (+1, param +64개)

[Part 4]  데이터: Oxford-IIIT Pet (같은 품종 이미지 3장)
  (4.1) 2장 concat : seq_len 400  (2×196+8)
        3장 concat : seq_len 596  (3×196+8)
  (4.2) front+side → KL divergence로 multi-view 효과 확인
  (4.3) single vs concat → embedding 평균, KL, token 분포 비교
